In [1]:
# ===== CHARGER DONNEES =====

import json

# Charger les exigences OpenAI
with open("srs_history_openai.json", "r", encoding="utf-8") as f:
    openai_generated_srs = json.load(f)

# Charger les exigences Gemini
with open("srs_history_gemini.json", "r", encoding="utf-8") as f:
    gemini_generated_srs = json.load(f)

# Exigences humaines
human_requirements = [
    "The system shall allow the administrator to create, modify, and delete teaching modules.",
    "The system shall allow one or more responsible teachers to be assigned to a module.",
    "The system shall allow the administrator to manage the database of teachers and students.",
    "A responsible teacher shall be able to add teachers or students to a module if they are already registered in the database.",
    "A teacher shall be able to record the absence of a student in a module.",
    "When recording an absence, a teacher shall be able to add a comment.",
    "A teacher shall be able to view the list of absences for students enrolled in their modules.",
    "Administrative staff shall be able to record the reason for an absence.",
    "Administrative staff shall be able to view all student absences.",
    "The system shall allow absences to be grouped by module or by student.",
    "The system shall alert teachers when a student has more than three unjustified absences.",
    "The system shall be accessible through a web browser.",
    "The system shall distinguish between roles: administrator, teacher, and administrative staff.",
    "Access to system functionalities shall be restricted according to the user’s role."
]

print("Loaded human requirements:", len(human_requirements))
print("Loaded openai_generated_srs:", len(openai_generated_srs))
print("Loaded gemini_generated_srs:", len(gemini_generated_srs))

In [2]:
# ===== NETTOYAGEE DE DONNEES =====

import re

def extract_requirements_clean(text):
    reqs = []
    if not isinstance(text, str):
        return reqs
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        line = re.sub(r"^Requirement\s+\d+\s*:\s*", "", line, flags=re.IGNORECASE)
        reqs.append(line)
    return reqs

def normalize(text):
    return text.lower().strip()

openai_generated_srs_cleaned = [extract_requirements_clean(gen["requirements"]) for gen in openai_generated_srs]
gemini_generated_srs_cleaned = [extract_requirements_clean(gen["requirements"]) for gen in gemini_generated_srs]

human_requirements = [normalize(r) for r in human_requirements]
openai_generated_srs_cleaned = [[normalize(r) for r in gen] for gen in openai_generated_srs_cleaned]
gemini_generated_srs_cleaned = [[normalize(r) for r in gen] for gen in gemini_generated_srs_cleaned]

print("Example cleaned openai SRS:")
for r in openai_generated_srs_cleaned[0][:3]:
    print("-", r)

In [3]:
# ===== CHARGER LES MODELES =====

%pip install sentence-transformers

import os
import json
import time
import urllib.request
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
ollama_model = os.environ.get("OLLAMA_MODEL", "llama3.2:1b")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# ===== FONCTIONS METRICS =====

def completeness_score(ai_reqs, human_reqs, threshold=0.7):
    ai_emb = embedding_model.encode(ai_reqs)
    human_emb = embedding_model.encode(human_reqs)
    sim_matrix = cosine_similarity(ai_emb, human_emb)
    matched_human = (sim_matrix >= threshold).any(axis=0).sum()
    return matched_human / len(human_reqs)

def _extract_json_object(text):
    if isinstance(text, dict):
        return text
    cleaned = text.strip()
    first = cleaned.find("{")
    last = cleaned.rfind("}")
    if first == -1 or last == -1:
        raise ValueError(f"Ollama did not return JSON: {text[:200]}")
    return json.loads(cleaned[first:last + 1])

def _call_ollama_json(prompt, max_retries=3):
    payload = {
        "model": ollama_model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": "json",
    }
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{ollama_base_url}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                response_data = json.loads(response.read().decode("utf-8"))
            content = response_data["message"]["content"]
            return _extract_json_object(content)
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

def evaluate_metric_with_ollama(metric_name, generated_requirements, human_requirements=None):
    # Note: Prompt building functions omitted for brevity but should be kept in your actual code
    return None